# 🍏 Observability & Tracing Demo with `azure-ai-projects` and OpenTelemetry 🍎

Welcome to this **Health & Fitness**-themed notebook, where we'll explore how to set up **observability** and **tracing** for:

1. **Basic LLM calls** using an `AIProjectClient` and its OpenAI-compatible client.
2. **Multi-step** interactions using an **Agent** (such as a Health Resource Agent).
3. **Tracing** your local usage in **console** (stdout) or via an **OTLP endpoint** (like **Prompty** or **Aspire**).
4. Sending those **traces** to **Azure Monitor** (Application Insights) so you can view them in **Microsoft Foundry**.

> **Disclaimer**: This is a fun demonstration of AI and observability! Any references to workouts, diets, or health routines in the code or prompts are purely for **educational** purposes. Always consult a professional for health advice.

## Contents
1. **Initialization**: Setting up environment, creating clients.
2. **Basic LLM Call**: Quick demonstration of retrieving model completions.
3. **Connections**: Listing project connections.
4. **Observability & Tracing**
   - **Console / Local** tracing
   - **Prompty / Aspire**: piping traces to a local OTLP endpoint
   - **Azure Monitor** tracing: hooking up to Application Insights
   - **Verifying** your traces in Microsoft Foundry
5. **Agent-based Example**:
   - Creating a simple "Health Resource Agent" referencing sample docs.
   - Multi-turn conversation with tracing.
   - Cleanup.

<img src="./seq-diagrams/1-observability.png" width="50%"/>

## 1. Initialization & Setup
**Prerequisites**:
- A `.env` file containing `PROJECT_ENDPOINT` (and optionally `MODEL_DEPLOYMENT_NAME`).
- Roles/permissions in Microsoft Foundry that let you do inference & agent creation.
- A local environment with `azure-ai-projects` (2.x), `openai`, and `opentelemetry` packages installed.

**What we do**:
- Load environment variables.
- Initialize the endpoint-based `AIProjectClient` and its OpenAI client.
- Check that we can talk to a model (like `gpt-5.4`).

In [1]:
import os
import sys
import time
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables
notebook_path = Path().absolute()
env_path = notebook_path.parent.parent / '.env'  # Adjust path as needed
load_dotenv(env_path)

# Initialize the endpoint-based AIProjectClient + its OpenAI client
try:
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Successfully created AIProjectClient + OpenAI client!")
except Exception as e:
    print(f"❌ Error creating AIProjectClient: {e}")

✅ Successfully created AIProjectClient + OpenAI client!


## 2. Basic LLM Call
We'll do a quick Responses API request to confirm everything is working. We'll ask a simple question: "How many feet are in a mile?"

In [2]:
try:
    # Default to "gpt-5.4" if no env var is set
    model_name = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4")

    user_question = "How many feet are in a mile?"
    response = openai_client.responses.create(
        model=model_name,
        input=user_question,
    )
    print("\nAnswer:")
    print(response.output_text)
    print("\nResponse status:", response.status)

except Exception as e:
    print("❌ Could not complete the response request:", e)


Answer:
There are **5,280 feet** in **1 mile**.

Response status: completed


## 3. List & Inspect Connections
Check out the **connections** your project has: these might be Azure OpenAI or other resource attachments. We'll just list them here for demonstration.

In [3]:
# List all connections in the project
all_conns = list(project_client.connections.list())
print(f"🔎 Found {len(all_conns)} total connections.")
for idx, c in enumerate(all_conns):
    print(f"{idx+1}) Name: {c.name}, Type: {c.type}, Target: {c.target}")

# Filter for Azure OpenAI connections (connection_type accepts the string category)
aoai_conns = list(project_client.connections.list(connection_type="AzureOpenAI"))
print(f"\n🌀 Found {len(aoai_conns)} Azure OpenAI connections:")
for c in aoai_conns:
    print(f"   -> {c.name}")

🔎 Found 3 total connections.
1) Name: aisearch2346625rdg92y, Type: ConnectionType.AZURE_AI_SEARCH, Target: https://aisearch2346625.search.windows.net/
2) Name: BingSearchrdg92y, Type: GroundingWithBingSearch, Target: https://api.bing.microsoft.com/
3) Name: insights234662rdg92y, Type: ConnectionType.APPLICATION_INSIGHTS, Target: /subscriptions/c0a71c49-0b19-4651-964d-79c6a13f26d1/resourceGroups/Foundry/providers/microsoft.insights/components/insights-234662

🌀 Found 0 Azure OpenAI connections:


# 4. Observability & Tracing

We want to **collect telemetry** from our LLM calls, for example:
- Timestamps of requests.
- Latency.
- Potential errors.
- Optionally, the actual prompts & responses (if you enable content recording).

We'll show how to set up:
1. **Console** or local OTLP endpoint instrumentation.
2. **Azure Monitor** instrumentation with Application Insights.
3. **Viewing** your traces in Azure AI Foundry's portal.

## 4.1 Local Console Debugging
We'll install instrumentation packages and enable them. Then we'll do a quick chat call to see if logs appear in **stdout**.

**Note**: If you want to see more advanced local dashboards, you can:
- Use [Prompty](https://github.com/microsoft/prompty).
- Use [Aspire Dashboard](https://learn.microsoft.com/dotnet/aspire/fundamentals/dashboard/standalone?tabs=bash) to visualize your OTLP traces.

In [4]:
# You only need to install these once.
!pip install opentelemetry-instrumentation-openai-v2 opentelemetry-exporter-otlp-proto-grpc


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


### 4.1.1 Enable OpenTelemetry for the OpenAI client
We instrument the **OpenAI** client (the one returned by `project_client.get_openai_client()`) so its chat/response calls emit OpenTelemetry spans. We:
1. Optionally capture **prompt & completion content** in traces.
2. Call `OpenAIInstrumentor().instrument()` to patch and enable the instrumentation.


In [5]:
import os
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor

# (Optional) capture prompt & completion contents in traces
os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"  # or 'false'

# Instrument the OpenAI client library so its calls emit OpenTelemetry spans.
# (We use the OpenAI client from project_client.get_openai_client(), so we
# instrument OpenAI rather than the retired azure-ai-inference client.)
OpenAIInstrumentor().instrument()
print("✅ OpenAI instrumentation enabled.")

✅ OpenAI instrumentation enabled.


### 4.1.2 Point Traces to Console or Local OTLP
The simplest option is to pipe spans to stdout. If you want to send them to Prompty or Aspire instead, specify the local OTLP endpoint URL, usually `http://localhost:4317` or similar.

In [6]:
from IPython.display import clear_output
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

# Pipe spans to stdout (console). To send to Prompty/Aspire instead, replace the
# ConsoleSpanExporter with an OTLPSpanExporter pointed at http://localhost:4317.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

try:
    user_prompt = "What's a simple 5-minute warmup routine?"
    local_resp = openai_client.responses.create(
        model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4"),
        input=user_prompt,
    )

    # Keep only relevant notebook output for readability.
    clear_output(wait=True)
    print("Local tracing test completed successfully.")
    print(f"Question: {user_prompt}")
    print(f"Response: {local_resp.output_text}")
except Exception as exc:
    clear_output(wait=True)
    print(f"Local tracing test failed: {exc}")

Local tracing test completed successfully.
Question: What's a simple 5-minute warmup routine?
Response: Here’s a simple **5-minute full-body warmup**:

### 5-Minute Warmup
**1. March in place – 1 minute**  
Lift your knees and swing your arms gently.

**2. Arm circles – 30 seconds forward, 30 seconds backward**  
Start small, then make the circles bigger.

**3. Bodyweight squats – 1 minute**  
Go slow and controlled. Don’t need to go super deep.

**4. Hip circles – 30 seconds each direction**  
Hands on hips, make smooth circles.

**5. Alternating lunges or step-backs – 1 minute**  
Keep your chest up and move at an easy pace.

**6. Light torso twists – 30 seconds**  
Relax your upper body and rotate side to side.

If you want, I can also give you:
- a **5-minute warmup before running**
- a **5-minute gym warmup**
- or a **beginner/no-jumping warmup**


## 4.2 Azure Monitor Tracing (Application Insights)
Now we'll set up tracing to **Application Insights**, which will forward your traces to the Microsoft Foundry **Observe → Tracing** page.

**Steps**:
1. In the Foundry portal, select **Observe → Tracing** for your project, then attach (or create) an **Application Insights** resource.
2. Copy that resource's **connection string** and set it as `APPLICATIONINSIGHTS_CONNECTION_STRING` in your `.env`.
3. Call `azure.monitor.opentelemetry.configure_azure_monitor(...)` with that connection string.
4. Make an inference call → traces appear on the Foundry **Observe → Tracing** page (and in Azure Monitor itself).

In [7]:
from IPython.display import clear_output
from azure.monitor.opentelemetry import configure_azure_monitor

# Set APPLICATIONINSIGHTS_CONNECTION_STRING in your .env; copy it from the
# Application Insights resource attached to your Foundry project's Tracing tab.
app_insights_conn_str = os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING")
if app_insights_conn_str:
    try:
        configure_azure_monitor(connection_string=app_insights_conn_str)

        # Test call that logs to Foundry Tracing via Azure Monitor.
        prompt_msg = "Any easy at-home cardio exercise recommendations?"
        response = openai_client.responses.create(
            model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4"),
            input=prompt_msg,
        )

        # Keep only relevant notebook output for readability.
        clear_output(wait=True)
        print("Azure Monitor tracing test completed successfully.")
        print(f"Question: {prompt_msg}")
        print(f"Response: {response.output_text}")
    except Exception as e:
        clear_output(wait=True)
        print(f"Azure Monitor tracing test failed: {e}")
else:
    clear_output(wait=True)
    print("Azure Monitor tracing is not configured.")
    print("Set APPLICATIONINSIGHTS_CONNECTION_STRING in your .env and run this cell again.")

Azure Monitor tracing test completed successfully.
Question: Any easy at-home cardio exercise recommendations?
Response: Yes — plenty. The best at-home cardio is usually the one that feels simple enough to actually do consistently.

## Easy at-home cardio ideas

### Low-impact / beginner-friendly
- **March in place**  
  Swing your arms and lift knees gently. Easy starting point.
- **Step touches**  
  Step side to side, adding arm movements.
- **Walking indoors**  
  Walk around your home, hallway, or follow a walking workout video.
- **Stair climbing**  
  Go up and down stairs at a comfortable pace.
- **Seated cardio**  
  Great if you want very low impact: seated punches, knee lifts, toe taps.
- **Dance**  
  Put on music and move however you like.
- **Shadow boxing**  
  Light punches, side steps, and upper-body movement.

### Moderate options
- **Bodyweight circuit**  
  Rotate through:
  - marching/high knees
  - squats to a chair
  - step jacks
  - mountain climbers on a wall o

{
    "name": "GET",
    "context": {
        "trace_id": "0x091d98f2f89906f57e90022320bd1f52",
        "span_id": "0x2185123597ee33d9",
        "trace_state": "[]"
    },
    "kind": "SpanKind.CLIENT",
    "parent_id": null,
    "start_time": "2026-08-09T19:49:40.615270Z",
    "end_time": "2026-08-09T19:49:41.109025Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "http.method": "GET",
        "http.url": "https://settings.sdk.monitor.azure.com/AzMonSDKDynamicConfigurationChanges?namespaces=python",
        "user_agent.original": "python-requests/2.34.2",
        "http.status_code": 200
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "bce50515-6dc5-4328-9b3e-94af55197e83",
            "service.name": "unknown_service"
        },
       

### 4.3 Viewing Traces in Microsoft Foundry
After running the above code:
1. Go to your Foundry project.
2. Select **Observe → Tracing** in the portal.
3. You should see the traces from your calls.
4. Filter, expand, or explore them as needed.

Also, if you want more advanced dashboards, you can open the **Application Insights** resource attached to your project. In the App Insights portal, you get additional features like **end-to-end transaction** details, log queries, and more.

# 5. Agent-based Example
We'll now create a Health Resource Agent that references sample docs about recipes or guidelines, then demonstrate:
1. Creating an agent with instructions.
2. Creating a conversation.
3. Running multi-step queries with observability enabled.
4. Optionally cleaning up resources at the end.

> The Microsoft Agent Framework approach is helpful when you want more sophisticated conversation flows or tool usage, such as file search.

## 5.1 Create Sample Files & Vector Store
We'll create dummy `.md` files about recipes/guidelines, then push them into a **vector store** so our agent can do semantic search.

(*This portion is a quick summary—see [the other file-search tutorial] if you need more details.)

In [11]:
def create_sample_files():
    """Create some local .md files with sample text."""
    recipes_md = """# Healthy Recipes Database

## Gluten-Free Recipes
1. Quinoa Bowl
   - Ingredients: quinoa, vegetables, olive oil
   - Instructions: Cook quinoa, add vegetables

2. Rice Pasta
   - Ingredients: rice pasta, mixed vegetables
   - Instructions: Boil pasta, saute vegetables

## Diabetic-Friendly Recipes
1. Low-Carb Stir Fry
   - Ingredients: chicken, vegetables, tamari sauce
   - Instructions: Cook chicken, add vegetables

## Heart-Healthy Recipes
1. Baked Salmon
   - Ingredients: salmon, lemon, herbs
   - Instructions: Season salmon, bake

2. Mediterranean Bowl
   - Ingredients: chickpeas, vegetables, tahini
   - Instructions: Combine ingredients
"""

    guidelines_md = """# Dietary Guidelines

## General Guidelines
- Eat a variety of foods
- Control portion sizes
- Stay hydrated

## Special Diets
1. Gluten-Free Diet
   - Avoid wheat, barley, rye
   - Focus on naturally gluten-free foods

2. Diabetic Diet
   - Monitor carbohydrate intake
   - Choose low glycemic foods

3. Heart-Healthy Diet
   - Limit saturated fats
   - Choose lean proteins
"""

    with open("recipes.md", "w", encoding="utf-8") as f:
        f.write(recipes_md)
    with open("guidelines.md", "w", encoding="utf-8") as f:
        f.write(guidelines_md)

    print("📄 Created sample resource files: recipes.md, guidelines.md")
    return ["recipes.md", "guidelines.md"]


sample_files = create_sample_files()


def create_vector_store(files, store_name="my_health_resources"):
    try:
        # Create the vector store, then upload and attach each file.
        vs = openai_client.vector_stores.create(name=store_name)
        uploaded_ids = []
        for fp in files:
            with open(fp, "rb") as fh:
                uploaded = openai_client.files.create(purpose="assistants", file=fh)
            openai_client.vector_stores.files.create_and_poll(vector_store_id=vs.id, file_id=uploaded.id)
            uploaded_ids.append(uploaded.id)
            print(f"✅ Uploaded: {fp} -> File ID: {uploaded.id}")

        print(f"🎉 Created vector store '{store_name}', ID: {vs.id}")
        return vs, uploaded_ids
    except Exception as e:
        print(f"❌ Error creating vector store: {e}")
        return None, []


vector_store, file_ids = None, []
if sample_files:
    vector_store, file_ids = create_vector_store(sample_files, store_name="health_resources_example")

📄 Created sample resource files: recipes.md, guidelines.md
✅ Uploaded: recipes.md -> File ID: assistant-DUdsz67zyMX4x4ffDdRg3P
✅ Uploaded: guidelines.md -> File ID: assistant-5Fx1Q27d3p321HX7JAv9RC
🎉 Created vector store 'health_resources_example', ID: vs_iSmMT4AqPe3IOrrl5UWCLKXd


## 5.2 Create a Health Resource Agent
We'll create a **FileSearchTool** referencing the vector store, then create an agent with instructions that it should:
1. Provide disclaimers.
2. Offer general nutrition or recipe tips.
3. Cite sources if possible.
4. Encourage professional consultation for deeper medical advice.


In [16]:
from IPython.display import clear_output
from azure.ai.projects.models import FileSearchTool, PromptAgentDefinition


def create_health_agent(vs_id):
    try:
        instructions = """
            You are a health resource advisor with access to dietary and recipe files.
            You:
            1. Always present disclaimers (you're not a medical professional)
            2. Provide references to files when possible
            3. Focus on general nutrition or recipe tips.
            4. Encourage professional consultation for more detailed advice.
        """

        agent = project_client.agents.create_version(
            agent_name="health-search-agent",
            definition=PromptAgentDefinition(
                model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4"),
                instructions=instructions,
                tools=[FileSearchTool(vector_store_ids=[vs_id])],
            ),
            description="Health resource search agent over uploaded documents.",
        )

        # Clear trace spam so the notebook shows only the user-friendly success line.
        clear_output(wait=True)
        print(f"🎉 Created agent '{agent.name}'")
        return agent
    except Exception as e:
        clear_output(wait=True)
        print(f"❌ Error creating health agent: {e}")
        return None


health_agent = None
if vector_store:
    health_agent = create_health_agent(vector_store.id)

🎉 Created agent 'health-search-agent'


## 5.3 Using the Agent
Let's create a new conversation and ask the agent some questions. We'll rely on the observability settings we already configured so each step is traced.

In [13]:
def ask_question(agent, conversation_id, user_question):
    try:
        print(f"User asked: '{user_question}'")
        # Run the agent via the Responses API (traced by the instrumentation above)
        response = openai_client.responses.create(
            conversation=conversation_id,
            input=user_question,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )
        print(f"Response status: {response.status}")
        return response
    except Exception as e:
        print(f"❌ Error asking question: {e}")
        return None

conversation = None
responses = []
if health_agent:
    conversation = openai_client.conversations.create()
    print(f"📝 Created new conversation, ID: {conversation.id}")
    # Let's ask a few sample questions
    queries = [
        "Could you suggest a gluten-free lunch recipe?",
        "Show me some heart-healthy meal ideas.",
        "What guidelines do you have for someone with diabetes?"
    ]
    for q in queries:
        r = ask_question(health_agent, conversation.id, q)
        if r:
            responses.append((q, r))

📝 Created new conversation, ID: conv_961cb86446a62f1d00jiqYu502MGtPm2zlMlCcw7m49tDWPvij
User asked: 'Could you suggest a gluten-free lunch recipe?'
Response status: completed
User asked: 'Show me some heart-healthy meal ideas.'
Response status: completed
User asked: 'What guidelines do you have for someone with diabetes?'
Response status: completed


### 5.3.1 Viewing the conversation
We can retrieve the conversation messages to see how the agent responded, check if it cited file passages, etc.

In [14]:
def display_results(pairs):
    try:
        print("\n🗣️ Conversation:")
        for user_question, response in pairs:
            print(f"[USER]: {user_question}")
            print(f"[ASSISTANT]: {response.output_text}\n")

            # Surface any file citations from the response annotations
            for item in (response.output or []):
                if getattr(item, "type", "") != "message":
                    continue
                for content in (getattr(item, "content", None) or []):
                    for ann in (getattr(content, "annotations", None) or []):
                        if getattr(ann, "type", "") == "file_citation":
                            print(f"   📎 Citation: {getattr(ann, 'filename', '') or getattr(ann, 'file_id', '')}")
    except Exception as e:
        print(f"❌ Could not display results: {e}")

# If we asked questions above, display the answers + citations
if responses:
    display_results(responses)


🗣️ Conversation:
[USER]: Could you suggest a gluten-free lunch recipe?
[ASSISTANT]: Sure — I’m not a medical professional, but I can suggest a simple gluten-free lunch idea based on your uploaded recipe and dietary guideline files. For a gluten-free diet, the guideline is to avoid wheat, barley, and rye and focus on naturally gluten-free foods .

Gluten-free lunch idea: Quinoa Bowl  
This recipe appears in your recipe file as a gluten-free option .

Ingredients:
- Quinoa
- Vegetables
- Olive oil 

Instructions:
1. Cook the quinoa.
2. Add vegetables.
3. Drizzle with olive oil and serve. 

A few easy ways to make it more filling:
- Add a protein like chickpeas, grilled chicken, or tofu
- Season with lemon juice, herbs, salt, and pepper
- Use roasted vegetables for extra flavor

Another gluten-free lunch option from your file is Rice Pasta with mixed vegetables: boil the rice pasta and sauté the vegetables .

If you want, I can also turn one of these into a more detailed 15-minute recipe

### 👀 See your agent live in the Foundry portal

Before cleaning up, confirm the agent now exists as a first-class resource in your Foundry project:

1. In a browser, open the [Microsoft Foundry portal](https://ai.azure.com) and select your project.
2. In the top navigation select **Build**, then select **Agents** in the left pane.
3. Locate **`health-search-agent`** in the list — the agent you just created and traced. Open it to inspect its instructions and its **File Search** tool.
4. Return to this notebook and run the next cell to delete the agent, vector store, files, and local samples.

# 6. Cleanup
If desired, we can remove the vector store, files, and agent to keep things tidy. (In a real solution, you might keep them around.)

In [15]:
from IPython.display import clear_output

def cleanup_resources():
    def safe_delete(action):
        try:
            action()
            return True
        except Exception:
            return False

    def add_message(message):
        if message:
            messages.append(message)

    messages = []

    agent = globals().get("health_agent")
    if agent:
        safe_delete(
            lambda: project_client.agents.delete_version(
                agent_name=agent.name,
                agent_version=agent.version,
            )
        )
        add_message(f"🗑️ Deleted health agent: {agent.name} (version: {agent.version})")

    store = globals().get("vector_store")
    if store:
        safe_delete(lambda: openai_client.vector_stores.delete(store.id))
        store_name = getattr(store, "name", "unknown")
        add_message(f"🗑️ Deleted vector store: {store_name} ({store.id})")

    uploaded_file_ids = globals().get("file_ids") or []
    if uploaded_file_ids:
        for file_id in uploaded_file_ids:
            safe_delete(lambda file_id=file_id: openai_client.files.delete(file_id))
        add_message(f"🗑️ Deleted uploaded files: {', '.join(uploaded_file_ids)}")

    local_files = globals().get("sample_files") or []
    deleted_local_files = []
    for file_name in local_files:
        if os.path.exists(file_name):
            os.remove(file_name)
            deleted_local_files.append(file_name)
    if deleted_local_files:
        add_message(f"🗑️ Deleted local sample files: {', '.join(deleted_local_files)}")

    clear_output(wait=True)
    if messages:
        print("\n".join(messages))
    else:
        print("No cleanup actions were needed.")


cleanup_resources()

🗑️ Deleted health agent: health-search-agent (version: 6)
🗑️ Deleted vector store: health_resources_example (vs_iSmMT4AqPe3IOrrl5UWCLKXd)
🗑️ Deleted uploaded files: assistant-DUdsz67zyMX4x4ffDdRg3P, assistant-5Fx1Q27d3p321HX7JAv9RC
🗑️ Deleted local sample files: recipes.md, guidelines.md


# 🎉 Wrap-Up
We've demonstrated:
1. **Basic LLM calls** with `AIProjectClient`.
2. **Listing connections** in your Azure AI Foundry project.
3. **Observability & tracing** in both local (console, OTLP endpoint) and cloud (App Insights) contexts.
4. A quick **Agent** scenario that uses a vector store for searching sample docs.

## Next Steps
- Check the **Tracing** tab in your Azure AI Foundry portal to see the logs.
- Explore advanced queries in Application Insights.
- Use [Prompty](https://github.com/microsoft/prompty) or [Aspire](https://learn.microsoft.com/dotnet/aspire/) for local telemetry dashboards.
- Incorporate this approach into your **production** GenAI pipelines!

> 🏋️ **Health Reminder**: The LLM's suggestions are for demonstration only. For real health decisions, consult a professional.

Happy Observing & Tracing! 🎉